In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

instructions = """

You are a course teaching assistant.
Answer the question based on CONTEXT from the FAQ Database.
Use only the facts from the CONTEXT when answering the question.
""".strip()

assistant = RAGBase(
    index= index,
    llm_client=openai_client,
    instructions=instructions,
)

In [ ]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

In [ ]:
messages = [
    {'role' : 'user', 'content': 'I just discovered the course, can I join it?'
    }
]

response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
)

response.output_text

In [ ]:
def search(query):
    boost_dict = {'question' : 3.0, 'section' : 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict= boost_dict,
        filter_dict= filter_dict
    )

In [ ]:
index.search("How to run ollama?")

In [ ]:
search_tool = {
    'type' : 'function',
    'name' : 'search',
    'description' : 'Search the FAQ database for entries matching the given query.',
    'parameters' : {
        'type' :'object',
        'properties' :{
            'query' : {
                'type' : 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required':['query'],
        'additionalProperties' : False
    }
}

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input = messages,
    tools = [search_tool]

)

In [ ]:
len(response.output)

In [ ]:
call = response.output[0]

In [ ]:
call

In [ ]:
import json

args = json.loads(call.arguments)

In [ ]:
results = search(**args)

In [ ]:
result_json = json.dumps(results, indent=2)

In [ ]:
print(result_json)

In [ ]:
function_call_output = {
    'type':'function_call_output',
    'call_id' : call.call_id,
    'output' : result_json
}

In [ ]:
messages.append(call)

In [ ]:
messages.append(function_call_output)

In [ ]:
messages

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [ ]:
print(response.output_text)

In [ ]:
usage = response.usage

usage.input_tokens, usage.output_tokens

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

In [ ]:
messages

In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
#encouraging multiple searchees
 
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

In [ ]:
#restricting off topic questions
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")